# ResNet18 binaire — Option A (loss pondérée) vs Option B (sampler)

Comparaison de 2 méthodes de gestion du déséquilibre (tumeur vs pas de tumeur).

In [8]:
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, recall_score, accuracy_score

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch: 2.10.0+cpu
CUDA: False


## 1) Dataset

Modifie `DATASET_ROOT` pour pointer sur le dossier qui contient `no_tumor/`, `glioma_tumor/`, etc.

In [9]:
import sys
from pathlib import Path

# Cherche la racine du projet (celle qui contient le dossier "src")
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # si tu as lancé Jupyter depuis /notebooks, on remonte d'un niveau
    PROJECT_ROOT = PROJECT_ROOT.parent

## 2) Split train/val/test (stratifié)

In [10]:
paths = [str(p) for p,_ in all_items]
y = np.array([lab for _,lab in all_items])

X_trainval, X_test, y_trainval, y_test = train_test_split(paths, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.15, random_state=42, stratify=y_trainval)

print("Train:", len(X_train), Counter(y_train))
print("Val:  ", len(X_val), Counter(y_val))
print("Test: ", len(X_test), Counter(y_test))

def make_items(X, y):
    return [(Path(p), int(lbl)) for p,lbl in zip(X,y)]

train_items = make_items(X_train, y_train)
val_items   = make_items(X_val, y_val)
test_items  = make_items(X_test, y_test)


NameError: name 'all_items' is not defined

## 3) Transforms + DataLoaders

In [ ]:
img_size = 224
train_tfms = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
eval_tfms = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = BrainMRIDatasetBinary(DATASET_ROOT, items=train_items, transform=train_tfms)
val_ds   = BrainMRIDatasetBinary(DATASET_ROOT, items=val_items, transform=eval_tfms)
test_ds  = BrainMRIDatasetBinary(DATASET_ROOT, items=test_items, transform=eval_tfms)

batch_size=32
num_workers=2
pin_memory = (device.type=="cuda")

val_loader  = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_ds,batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)


## 4) Boucles train/eval + modèle ResNet18

In [ ]:
from tqdm.auto import tqdm

def build_resnet18_binary():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred=np.asarray(y_pred)
    return {
        "accuracy": accuracy_score(y_true,y_pred),
        "recall_tumor": recall_score(y_true,y_pred,pos_label=1, zero_division=0),
        "f1_tumor": f1_score(y_true,y_pred,pos_label=1, zero_division=0),
        "cm": confusion_matrix(y_true,y_pred,labels=[0,1])
    }

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total=0.0; n=0
    for xb,yb in tqdm(loader, leave=False):
        xb=xb.to(device, non_blocking=True)
        yb=yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits=model(xb)
            loss=criterion(logits,yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        bs=xb.size(0)
        total += loss.item()*bs
        n += bs
    return total/max(n,1)

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total=0.0; n=0
    yp=[]; yt=[]
    for xb,yb in loader:
        xb=xb.to(device, non_blocking=True)
        yb=yb.to(device, non_blocking=True)
        logits=model(xb)
        loss=criterion(logits,yb)
        bs=xb.size(0)
        total += loss.item()*bs
        n += bs
        pred=torch.argmax(logits, dim=1)
        yp.append(pred.cpu().numpy())
        yt.append(yb.cpu().numpy())
    yp=np.concatenate(yp); yt=np.concatenate(yt)
    return total/max(n,1), metrics(yt, yp)

def fit(model, train_loader, criterion, optimizer, epochs=5):
    scaler=torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
    hist=[]
    best_f1=-1; best_state=None
    for ep in range(1, epochs+1):
        tr_loss=train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_m=eval_epoch(model, val_loader, criterion)
        hist.append((tr_loss, va_loss, va_m["accuracy"], va_m["recall_tumor"], va_m["f1_tumor"]))
        print(f"Epoch {ep}/{epochs} | tr={tr_loss:.4f} | va={va_loss:.4f} | acc={va_m['accuracy']:.4f} | rec1={va_m['recall_tumor']:.4f} | f1_1={va_m['f1_tumor']:.4f}")
        if va_m["f1_tumor"]>best_f1:
            best_f1=va_m["f1_tumor"]
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return hist, best_f1


## 5) Option A — Loss pondérée

In [ ]:
train_loader_A = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)

counts = Counter([lbl for _,lbl in train_items])
n0,n1 = counts[0], counts[1]
w0 = (n0+n1)/(2*n0); w1=(n0+n1)/(2*n1)
class_w = torch.tensor([w0,w1], dtype=torch.float32).to(device)
print("weights A:", class_w.tolist(), "counts:", counts)

criterion_A = nn.CrossEntropyLoss(weight=class_w)

model_A = build_resnet18_binary().to(device)
for p in model_A.parameters(): p.requires_grad=False
for p in model_A.fc.parameters(): p.requires_grad=True

optimizer_A = torch.optim.AdamW(filter(lambda p:p.requires_grad, model_A.parameters()), lr=3e-4, weight_decay=1e-4)

hist_A, bestA = fit(model_A, train_loader_A, criterion_A, optimizer_A, epochs=5)
test_loss_A, test_m_A = eval_epoch(model_A, test_loader, criterion_A)
print("TEST A:", {k:v for k,v in test_m_A.items() if k!='cm'})
print("CM A:\n", test_m_A["cm"])


## 6) Option B — WeightedRandomSampler

In [ ]:
train_labels = np.array([lbl for _,lbl in train_items])
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)
train_loader_B = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=pin_memory)

criterion_B = nn.CrossEntropyLoss()

model_B = build_resnet18_binary().to(device)
for p in model_B.parameters(): p.requires_grad=False
for p in model_B.fc.parameters(): p.requires_grad=True

optimizer_B = torch.optim.AdamW(filter(lambda p:p.requires_grad, model_B.parameters()), lr=3e-4, weight_decay=1e-4)

hist_B, bestB = fit(model_B, train_loader_B, criterion_B, optimizer_B, epochs=5)
test_loss_B, test_m_B = eval_epoch(model_B, test_loader, criterion_B)
print("TEST B:", {k:v for k,v in test_m_B.items() if k!='cm'})
print("CM B:\n", test_m_B["cm"])


## 7) Comparaison

In [ ]:
import pandas as pd
pd.DataFrame([
    {"option":"A loss pondérée", "accuracy": test_m_A["accuracy"], "recall_tumor": test_m_A["recall_tumor"], "f1_tumor": test_m_A["f1_tumor"]},
    {"option":"B sampler", "accuracy": test_m_B["accuracy"], "recall_tumor": test_m_B["recall_tumor"], "f1_tumor": test_m_B["f1_tumor"]},
])
